# TAURON (TPE) — regresja względem spółek GPW

Projekt ekonometryczny: model liniowy log-stop zwrotu TAURON-a na podstawie 4 najsilniej skorelowanych spółek z GPW.

**Etapy:**
1. Wczytanie danych (stooq, format `.txt`) i obliczenie log-stop zwrotu
2. Wybór zmiennych objasniających wg korelacji
3. Test ADF + usunięcie outlierów
4. Wizualizacja i test liniowości
5. Współliniowość: Farrar-Glauber, Haitovsky, VIF
6. OLS
7. Diagnostyka reszt (DW, Breusch-Pagan, ACF, Q-Q)
8. Prognoza out-of-sample (ostatnie 10 %)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.stats.api as sms
from IPython.display import display
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller

DATA_DIR = Path('data')
MAIN_TICKER_FILE = DATA_DIR / 'tpe.txt'
TOP_N = 4
OUTLIER_THRESHOLD = 0.05

## 1) Wczytanie danych i log-stopy zwrotu

In [ ]:
def load_log_returns(path: Path):
    if path.stat().st_size == 0:
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if '<DATE>' not in df.columns or '<CLOSE>' not in df.columns or len(df) < 2:
        return None
    df['<DATE>'] = pd.to_datetime(df['<DATE>'], format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['<DATE>']).set_index('<DATE>').sort_index()
    price = df['<CLOSE>'].astype(float)
    return np.log(price).diff().rename(path.stem.upper())


tauron_returns = load_log_returns(MAIN_TICKER_FILE).dropna().rename('TAURON')

other_returns = []
for f in sorted(DATA_DIR.iterdir()):
    if f.suffix.lower() != '.txt' or f.name == MAIN_TICKER_FILE.name:
        continue
    s = load_log_returns(f)
    if s is not None:
        other_returns.append(s.dropna())

returns_df = pd.concat([tauron_returns] + other_returns, axis=1)
returns_df = returns_df.dropna(axis=1, thresh=250)
print('returns_df shape (before row dropna):', returns_df.shape)
returns_df.head()

## 2) Wybór zmiennych wg korelacji

In [ ]:
corr_with_tauron = (
    returns_df.corr(min_periods=100)['TAURON']
    .abs()
    .sort_values(ascending=False)
    .dropna()
)
print('Ranking korelacji z TAURON-em (top 8):')
print(corr_with_tauron.head(8))

selected_vars = corr_with_tauron.drop('TAURON').index[:TOP_N].tolist()
print('\nWybrane zmienne objasniajace:', selected_vars)

# Zawezamy do 5 kolumn i usuwamy braki
returns_df = returns_df[['TAURON'] + selected_vars].dropna()
print('returns_df shape (5 cols, no NaN):', returns_df.shape)

## 3) Test Augmented Dickey-Fuller (ADF)

In [ ]:
def ensure_stationary(series, name, alpha=0.05):
    pvalue = adfuller(series, autolag='AIC')[1]
    if pvalue > alpha:
        print(f'{name}: niestacjonarny (p={pvalue:.3f}) -> roznicowanie')
        return series.diff().dropna().rename(name)
    print(f'{name}: stacjonarny (p={pvalue:.3f})')
    return series.rename(name)


stationary = {'TAURON': ensure_stationary(returns_df['TAURON'], 'TAURON')}
for var in selected_vars:
    stationary[var] = ensure_stationary(returns_df[var], var)

stationary_df = pd.concat(stationary.values(), axis=1).dropna()
print('stationary_df shape:', stationary_df.shape)
stationary_df.head()

### 3a) Usunięcie outlierów |r| > 5 %

In [ ]:
mask = (stationary_df.abs() <= OUTLIER_THRESHOLD).all(axis=1)
stationary_df = stationary_df.loc[mask]
print(f'Po usunieciu obserwacji |r| > {OUTLIER_THRESHOLD * 100:.1f}%: {stationary_df.shape[0]} wierszy')

## 4) Wizualizacja i korelacje

In [ ]:
stationary_df[selected_vars].plot(subplots=True, figsize=(8, 6), marker='o', linestyle='-')
plt.suptitle('Stacjonarne serie: top 4 zmienne', y=1.02)
plt.tight_layout()
plt.show()

stationary_df.plot(subplots=True, figsize=(10, 8), marker='o', linestyle='-')
plt.suptitle('Stacjonarne serie: TAURON + top 4', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr_stat = stationary_df.corr()
print('Macierz korelacji (po stacjonaryzacji):')
print(corr_stat)

plt.figure(figsize=(6, 5))
sns.heatmap(corr_stat, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Macierz korelacji - dane stacjonarne')
plt.show()

### 4a) Test liniowości

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, var in zip(axes.ravel(), selected_vars):
    sns.regplot(
        x=stationary_df[var],
        y=stationary_df['TAURON'],
        ax=ax,
        scatter_kws=dict(alpha=0.4),
        line_kws=dict(color='red'),
    )
    ax.set_xlabel(f'{var} (log returns)')
    ax.set_ylabel('TAURON (log returns)')
    ax.set_title(f'TAURON vs {var}')
plt.tight_layout()
plt.suptitle('Sprawdzenie liniowosci', y=1.02)
plt.show()

## 5) Współliniowość: Farrar-Glauber & Haitovsky

In [ ]:
R = stationary_df[selected_vars].corr()
det_R = np.linalg.det(R)
n_obs = stationary_df.shape[0]
k_vars = len(selected_vars)
df_chi = k_vars * (k_vars - 1) / 2

FG_stat = -(n_obs - 1 - (2 * k_vars + 5) / 6) * np.log(det_R)
FG_pvalue = 1 - stats.chi2.cdf(FG_stat, df_chi)

H_stat = -(n_obs - (2 * k_vars + 11) / 6) * np.log(det_R)
H_pvalue = 1 - stats.chi2.cdf(H_stat, df_chi)

multi_df = pd.DataFrame({
    'Determinant':            [det_R],
    'Log Determinant':        [np.log(det_R)],
    'Log (1 - Determinant)':  [np.log(1 - det_R)],
    'N':                      [n_obs],
    'k':                      [k_vars],
    'Degrees of Freedom':     [df_chi],
    'Farrar-Glauber stat':    [FG_stat],
    'Haitovsky stat':         [H_stat],
    'Farrar-Glauber p-value': [FG_pvalue],
    'Haitovsky p-value':      [H_pvalue],
})
display(multi_df)

## 6) Regresja OLS

In [ ]:
Y = stationary_df['TAURON']
X = sm.add_constant(stationary_df[selected_vars])
model = sm.OLS(Y, X).fit()
print(model.summary())

params = model.params
formula = f"TAURON_t = {params['const']:.6f}"
for var in selected_vars:
    formula += f' + ({params[var]:.6f})*{var}_t'
print('\nRownanie regresji:\n', formula)

## 7) Diagnostyka reszt

In [ ]:
resid = model.resid
fitted = model.fittedvalues
exog = model.model.exog
names = model.model.exog_names

vif = pd.Series(
    [variance_inflation_factor(exog, i) for i in range(exog.shape[1])],
    index=names,
)
print('Variance Inflation Factors:')
display(vif)

print('Durbin-Watson:', sm.stats.stattools.durbin_watson(resid))
print('Breusch-Pagan p-value:', sms.het_breuschpagan(resid, exog)[1])

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(14, 10))
gs = fig.add_gridspec(3, 2)

ax1 = fig.add_subplot(gs[0, 0])
plot_acf(resid, ax=ax1, lags=40, title='ACF reszt')

ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(resid, bins=30, alpha=0.6)
ax2.axvline(0, linestyle='--', color='grey')
ax2.set_title('Histogram reszt')

ax3 = fig.add_subplot(gs[1, :])
ax3.scatter(fitted, resid, alpha=0.4)
ax3.axhline(0, linestyle='--', color='grey')
ax3.set_xlabel('Wartosci dopasowane')
ax3.set_ylabel('Reszty')
ax3.set_title('Reszty vs dopasowane')

ax4 = fig.add_subplot(gs[2, 0])
sm.qqplot(resid, line='45', ax=ax4, fit=True)
ax4.set_title('QQ plot reszt')

ax5 = fig.add_subplot(gs[2, 1])
colors = plt.cm.viridis(np.linspace(0, 1, k_vars))
for c, var in zip(colors, selected_vars):
    ax5.scatter(stationary_df[var], resid, alpha=0.4, label=var, color=c)
ax5.axhline(0, linestyle='--', color='grey')
ax5.set_title('Reszty vs zmienne objasniajace')
ax5.legend(frameon=False, fontsize=8)

plt.show()

## 8) Prognoza (ostatnie 10 % obserwacji)

In [ ]:
n_forecast = int(0.1 * len(stationary_df))
train, test = stationary_df.iloc[:-n_forecast], stationary_df.iloc[-n_forecast:]
model_train = sm.OLS(train['TAURON'], sm.add_constant(train[selected_vars])).fit()

y_pred = model_train.predict(sm.add_constant(test[selected_vars]))
y_true = test['TAURON']

MAE = np.mean(np.abs(y_pred - y_true))
RMSE = np.sqrt(np.mean((y_pred - y_true) ** 2))
SMAPE = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true))) * 100

print(f'MAE   = {MAE:.6f}')
print(f'RMSE  = {RMSE:.6f}')
print(f'sMAPE = {SMAPE:.2f}%')

forecast_df = pd.DataFrame({'Actual': y_true, 'Forecast': y_pred})
display(forecast_df.head())

plt.figure(figsize=(10, 4))
plt.plot(test.index, y_true, label='Rzeczywiste')
plt.plot(test.index, y_pred, label='Prognoza', alpha=0.7)
plt.legend()
plt.title('TAURON: rzeczywiste vs prognoza (dziennie)')
plt.show()